In [ ]:
!pip install -q openai python-dotenv

In [ ]:
import os
import json
import re
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI

In [ ]:
load_dotenv()

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
print(f"API key loaded: {'Yes' if OPENROUTER_API_KEY else 'No'}")

In [ ]:
def read_file(path):
    """Read a file and return its contents."""
    return Path(path).read_text()

read_file_schema = {
    "type": "function",
    "function": {
        "name": "read_file",
        "description": "Read the contents of a file.",
        "parameters": {
            "type": "object",
            "properties": {
                "path": {"type": "string", "description": "Path to the file"}
            },
            "required": ["path"]
        }
    }
}

In [ ]:
def glob_files(pattern):
    """Find files matching a glob pattern."""
    return [str(p) for p in Path('.').glob(pattern)]

glob_schema = {
    "type": "function",
    "function": {
        "name": "glob",
        "description": "Find files matching a glob pattern like **/*.py or data/**/*.py",
        "parameters": {
            "type": "object",
            "properties": {
                "pattern": {"type": "string", "description": "Glob pattern"}
            },
            "required": ["pattern"]
        }
    }
}

In [ ]:
def grep_files(pattern, path):
    """Search file contents with regex. Returns list of (file, line_num, line) tuples."""
    regex = re.compile(pattern)
    results = []
    for f in Path('.').glob(path):
        if f.is_file():
            try:
                for i, line in enumerate(f.read_text().splitlines(), 1):
                    if regex.search(line):
                        results.append((str(f), i, line.strip()))
            except (UnicodeDecodeError, PermissionError):
                pass
    return results

grep_schema = {
    "type": "function",
    "function": {
        "name": "grep",
        "description": "Search file contents with regex. Returns matching lines with file paths and line numbers.",
        "parameters": {
            "type": "object",
            "properties": {
                "pattern": {"type": "string", "description": "Regex pattern to search for"},
                "path": {"type": "string", "description": "Glob pattern for files to search (e.g. **/*.py)"}
            },
            "required": ["pattern", "path"]
        }
    }
}

In [ ]:
TOOLS = [read_file_schema, glob_schema, grep_schema]

TOOL_MAP = {
    "read_file": read_file,
    "glob": glob_files,
    "grep": grep_files,
}

print(f"Tools registered: {list(TOOL_MAP.keys())}")

In [ ]:
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY,
)

MODEL = "nvidia/nemotron-3-ultra-550b-a55b:free"
TARGET_DIR = "data"

print(f"Model: {MODEL}")
print(f"Target: {TARGET_DIR}")

In [ ]:
TASK = f"""
Scan the codebase at {TARGET_DIR} and find all TODO and FIXME comments.

For each match, report:
- File path
- Line number
- The comment text
- A brief note on what the comment is about

Organize the results as a markdown summary grouped by file.
"""

In [ ]:
messages = [
    {"role": "system", "content": "You are a code exploration assistant. Scan codebases, find patterns, and produce structured markdown reports. Be thorough but concise. Always cite file paths and line numbers."},
    {"role": "user", "content": TASK},
]

tool_call_count = 0
MAX_ITERATIONS = 15
final_answer = None

for i in range(MAX_ITERATIONS):
    response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        tools=TOOLS,
    )

    if not response.choices:
        print(f"API returned no choices. Response: {response}")
        break

    choice = response.choices[0]
    message = choice.message

    if message.tool_calls:
        messages.append(message)

        for tool_call in message.tool_calls:
            func_name = tool_call.function.name
            func_args = json.loads(tool_call.function.arguments)
            tool_call_count += 1

            print(f"  [Call {tool_call_count}] {func_name}({func_args})")

            if func_name in TOOL_MAP:
                result = TOOL_MAP[func_name](**func_args)
            else:
                result = f"Unknown tool: {func_name}"

            result_str = json.dumps(result, default=str)

            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": result_str,
            })
    else:
        final_answer = message.content
        break

print(f"\nDone. Tool calls made: {tool_call_count}")

In [ ]:
print(f"{'='*50}")
print("AGENT RESPONSE:")
print(f"{'='*50}\n")
print(final_answer)

In [ ]:
todo_count = len(re.findall(r'(?i)TODO', final_answer))
fixme_count = len(re.findall(r'(?i)FIXME', final_answer))

file_mentions = len(set(re.findall(r'\b[\w/]+\.py\b', final_answer)))

print("="*50)
print("RESULT SUMMARY")
print("="*50)
print(f"  TODO mentions:    {todo_count}")
print(f"  FIXME mentions:   {fixme_count}")
print(f"  Unique files:     {file_mentions}")
print("="*50)

In [ ]:
judge_prompt = f"""
You are an evaluation judge. Analyze the following agent output and the codebase.

AGENT OUTPUT:
{final_answer}

TASK: Find all TODO and FIXME comments in the codebase.

Evaluate on these criteria:
1. COVERAGE: Did the agent find all the TODO/FIXME comments?
2. ACCURACY: Are all reported items real TODO/FIXME comments (not false positives from strings)?
3. COMPLETENESS: Did it include file paths and line numbers?
4. FORMAT: Is the output well-organized and readable?

Score each criterion 1-5 and give an overall score. Be strict.
"""

judge_response = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": judge_prompt}],
)

if judge_response.choices:
    judge_content = judge_response.choices[0].message.content
else:
    judge_content = "(No response from judge - API returned no choices)"

print("="*50)
print("LLM JUDGE EVALUATION")
print("="*50)
print(judge_content)